# InsPLAD training on Colab

Trains the powerline-inspection-demo models. Before running: **Runtime → Change runtime type → GPU** (L4 for the 640 baseline and the classifier; A100 for the 1280 / YOLO11-m runs).

Checkpoints go to Google Drive, so a disconnected session loses nothing: rerun the setup cells, rerun the same training cell, and it resumes from `last.pt` automatically.

In [ ]:
# 1. GPU check + persistent storage for checkpoints
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/powerline-runs'

## 2. Get the code

Preferred: clone from GitHub (fill in the URL once the repo is pushed). Fallback: upload a zip of the repo (without `data/`) to Drive as `powerline-inspection-demo.zip` and use the unzip line instead.

In [ ]:
REPO_URL = 'https://github.com/Josh-E-S/powerline-inspection-demo.git'

![ -d /content/powerline-inspection-demo ] || git clone $REPO_URL /content/powerline-inspection-demo
%cd /content/powerline-inspection-demo

## 3. Dataset

Pulled straight from the authors' Mendeley Data deposit (6.4 GB; datacenter bandwidth makes this a few minutes). Lands on Colab's fast local disk, which is also why prep's symlinks work: Drive mounts don't support them.

In [ ]:
!mkdir -p data/raw
!curl -L -C - -o /content/InsPLAD_Dataset.zip "https://data.mendeley.com/public-files/datasets/5n3fjgvfyz/files/96707044-99bb-40b2-bf23-6fa1b41ab9b0/file_downloaded"
!cd data/raw && unzip -q -o /content/InsPLAD_Dataset.zip InsPLAD-det.zip supervised_fault_classification.zip \
  && unzip -q -o InsPLAD-det.zip && unzip -q -o supervised_fault_classification.zip && rm *.zip
!ls data/raw

In [ ]:
# 4. Install deps (torch/torchvision are preinstalled on Colab) and stage the data
!pip install -q ultralytics onnx onnxruntime
!python3 scripts/prep_insplad.py

## 5. Smoke test first (~5 min total)

Tiny runs that exercise the entire pipeline: data loading, labels, training loop, checkpoints landing on Drive. If both finish and the `find` line lists `.pt` files under `powerline-runs/`, the full runs are safe to start. The point is to find path and permission mistakes in five minutes instead of thirty.

In [ ]:
!python3 scripts/train_detector.py --smoke --runs-dir $RUNS
!python3 scripts/train_classifier.py --smoke --runs-dir $RUNS
!find $RUNS -name '*.pt' | head

## 6. Detector runs

Run order per the spec's model quality plan: baseline first (proves the pipeline end to end), then the accuracy pushes. Each is resumable; rerun the same cell after a disconnect.

In [ ]:
# 6a. Baseline: YOLO11-s @ 640 (~1.5 h on L4)
!python3 scripts/train_detector.py --runs-dir $RUNS

In [ ]:
# 6b. Resolution push: s @ 1280 (prefer A100)
# !python3 scripts/train_detector.py --imgsz 1280 --runs-dir $RUNS

# 6c. Accuracy ceiling: m @ 1280 (A100)
# !python3 scripts/train_detector.py --model yolo11m.pt --imgsz 1280 --runs-dir $RUNS

# 6d. Rare-class oversampling variant
# !python3 scripts/train_detector.py --imgsz 1280 --oversample --runs-dir $RUNS

## 7. Condition classifier

EfficientNetV2-S on the 11 asset__condition classes; selects on val balanced accuracy. Fine on L4 (~30–45 min).

In [ ]:
!python3 scripts/train_classifier.py --runs-dir $RUNS

## 8. After training

Weights are on Drive under `powerline-runs/`:
- detector: `detect/<run-name>/weights/best.pt` (+ Ultralytics metrics/plots alongside)
- classifier: `classify/cls_effv2s/best.pt` (+ `metrics.jsonl`)

Download `best.pt` files locally for the export/quantization step (`scripts/export_quantize.py`). Record the GPU model shown by `nvidia-smi` for the README results table.

In [ ]:
# Quick look at detector training curves for a finished run
RUN_NAME = 'det_yolo11s_640'
from IPython.display import Image, display
display(Image(f'{RUNS}/detect/{RUN_NAME}/results.png'))